# BBC News: Tech Category Sub-classification
## NLP Pipeline: BGE-M3 Embeddings + BERTopic (UMAP + HDBSCAN + c-TF-IDF)

**Architecture:**
- Layer 1: Data Ingestion — Raw .txt files → List[String]
- Layer 2: BGE-M3 Embeddings — 8192 token context, 1024 dims
- Layer 3: UMAP — Non-linear reduction 1024 → 5 dims
- Layer 4: HDBSCAN — Automatic density-based clustering
- Layer 5a: CountVectorizer — Removes noise words per cluster
- Layer 5b: c-TF-IDF — Distinctive keywords across clusters
- Layer 6: KeyBERTInspired — Semantic keyword refinement
- Layer 7: Output — DataFrame [filename, category, sub_category, confidence_score, preview]

**Dataset:** BBC News Tech Category 511 raw articles
**Final Output:** DataFrame mapping every article to its sport sub-category

## Environment Setup & Imports
### Loading all required libraries for the full BERTopic pipeline

In [ ]:
# Core libraries
import os
import re
import pickle
import numpy as np
import pandas as pd
from loader import load_data

# Sentence Transformers — BGE-M3
from sentence_transformers import SentenceTransformer

# BERTopic pipeline
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

print("All imports successful")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## Data Ingestion Layer
### Loading raw Tech articles from local storage
### Source: BBC News dataset for business category
### No aggressive cleaning at this stage, BGE-M3 needs full sentence structure

In [ ]:
import os

# Path to Tech articles in local storage
data_path = '../data/tech'

# Load all txt files
Tech_docs = []
Tech_files = []

for filename in sorted(os.listdir(data_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(data_path, filename)
        with open(file_path, encoding='utf-8', errors='ignore') as f:
            text = f.read()
        Tech_docs.append(text)
        Tech_files.append(filename)

print(f"Total Tech articles loaded: {len(Tech_docs)}")
print(f"\nSample filename: {Tech_files[0]}")
print(f"\nSample article preview:")
print(f"{Tech_docs[0][:300]}")

## Data Normalization Layer
### Light cleaning only preserving full sentence structure for BGE-M3
### We only remove: encoding errors, excessive whitespace, HTML tags
### Punctuation, proper nouns, stopwords intentionally preserved

In [ ]:
def light_clean(text):
    # Fix encoding errors
    text = text.encode('utf-8', 'ignore').decode('utf-8')

    # Remove HTML tags if present
    text = re.sub(r'<.*?>', '', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply light cleaning
cleaned_Tech = [light_clean(doc) for doc in Tech_docs]

print(f"Cleaning complete — {len(cleaned_Tech)} articles processed")
print(f"\nBefore: {Tech_docs[0][:200]}")
print(f"\nAfter: {cleaned_Tech[0][:200]}")

##  Deduplication
### Removing duplicate articles before embedding generation
### Identical documents produce identical vectors hurts clustering quality
### Preserving filename mapping for final DataFrame output

In [ ]:
# Remove duplicates while preserving filename mapping
seen = set()
unique_Tech = []
unique_files = []

for doc, filename in zip(cleaned_Tech, Tech_files):
    if doc not in seen:
        seen.add(doc)
        unique_Tech.append(doc)
        unique_files.append(filename)

print(f"Before deduplication: {len(cleaned_Tech)} articles")
print(f"After deduplication:  {len(unique_Tech)} articles")
print(f"Duplicates removed:   {len(cleaned_Tech) - len(unique_Tech)}")

## Embedding Layer (BGE-M3)
### Model: BAAI/bge-m3 —> 8192 token context window, 1024 dimensional output
### Full-sequence self-attention transformer from BAAI
### Every article processed as one complete unit, no truncation
### Produces dense semantic vectors capturing meaning, context and intent

In [ ]:
# Load BGE-M3 model
print("Loading BGE-M3 model...")
model = SentenceTransformer('BAAI/bge-m3')
print("BGE-M3 loaded successfully")
print(f"Max sequence length: {model.max_seq_length}")

##  Generate BGE-M3 Document Embeddings
### Each article is encoded as a single 1024-dimensional semantic vector
### BGE-M3 reads the entire document using full-sequence self-attention
### batch_size=4 processes 4 articles at a time to manage memory efficiently
### show_progress_bar=True lets us track encoding progress

In [ ]:
print("Generating BGE-M3 embeddings for Tech articles...")
print("This will take a few minutes...")

embeddings = model.encode(
    unique_Tech,
    batch_size=4,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generated successfully")
print(f"Shape: {embeddings.shape}")
print(f"Each article \u2192 {embeddings.shape[1]} dimensional vector")

## BERTopic Pipeline (All 6 Modules)
### Module 3: UMAP — compresses 1024 dims to 5, random_state=42 for stability
### Module 4: HDBSCAN — finds natural sub-categories automatically
### Module 5a: CountVectorizer — removes noise words before keyword extraction
### Module 5b: c-TF-IDF — finds distinctive keywords per cluster
### Module 6: KeyBERTInspired — refines keywords using semantic similarity

In [ ]:
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN


# Module 3 — UMAP
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.01,
    random_state=42
)

# Module 4 — HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # Adjusted to identify more, smaller clusters
    min_samples=3,        # Maintained to allow for more fine-grained clusters
    metric='euclidean',
    cluster_selection_method='eom'
)

# Module 5a — CountVectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    max_df=0.8,
    ngram_range=(1, 2)
)

# Module 5b — c-TF-IDF
ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True
)

# Module 6 — KeyBERTInspired
representation_model = KeyBERTInspired()

# Assemble full BERTopic pipeline
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    verbose=True
)

print("BERTopic pipeline assembled successfully")
print("All 6 modules configured")

## Fit BERTopic to BGE-M3 Embeddings
### Passing pre-computed BGE-M3 embeddings directly into BERTopic
### BERTopic runs all 6 modules sequentially:
### UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERTInspired
### topics_ contains sub-category assignment for every article
### probs_ contains confidence score for each assignment

In [ ]:
# Fit BERTopic using pre-computed BGE-M3 embeddings
print("Running BERTopic pipeline...")
print("UMAP → HDBSCAN → Vectorizer → c-TF-IDF → KeyBERT")
print("This may take a few minutes...")

topics, probs = topic_model.fit_transform(
    unique_Tech,
    embeddings=embeddings
)

print(f"\nBERTopic complete")
print(f"Total articles processed: {len(topics)}")
print(f"Sub-categories discovered: {len(set(topics)) - 1}")
print(f"Noise articles (topic -1): {topics.count(-1)}")

### Storage: Final DataFrame

The final DataFrame from the architecture. It  includes the original filename, the main category ('Tech'), the assigned sub-category (from BERTopic), and a preview of the article content.

In [ ]:
final_df = pd.DataFrame({
    'filename': unique_files,
    'category': 'Tech',
    'sub_category_id': topics,
    'preview': [doc[:300] + '...' for doc in unique_Tech]
})

# Get human-readable topic names for sub-categories
topic_names = topic_model.get_topic_info()['Name']
# Create a mapping from topic ID to topic name
topic_id_to_name = dict(zip(topic_model.get_topic_info().Topic, topic_model.get_topic_info().Name))

# Map sub_category_id to sub_category_name
final_df['sub_category_name'] = final_df['sub_category_id'].map(topic_id_to_name)

# Handle noise articles (-1 topic_id) as 'Noise' or 'Unassigned'
final_df['sub_category_name'] = final_df['sub_category_name'].fillna('Noise')

# Reorder and select final columns
final_df = final_df[['filename', 'category', 'sub_category_id', 'sub_category_name', 'preview']]

print(f"Final DataFrame created with {len(final_df)} entries.")
display(final_df)

### Storage: Final DataFrame

Now, let's create the final DataFrame as specified in your architecture. It will include the original filename, the main category ('Tech'), the assigned sub-category (human labelling ), and a preview of the article content.

In [ ]:
# TECH FINAL DATAFRAME WITH NOISE RELABELLING (Step 7)


# Consolidation — merge all gaming topics into one
tech_consolidation = {
    4: 2,   # Game reviews/awards → Gaming Industry
    6: 2,   # Handheld consoles  → Gaming Industry
}

tech_labels = {
    0: "Digital Technology",
    1: "Cybersecurity Online",
    2: "Gaming Industry",
    3: "Internet Society",
    5: "Digital Piracy",
    7: "Software Patents",
    -1: "Unassigned"
}

# Noise remap — 14 articles
tech_noise_remap = {
    # Gaming Industry (2)
    "015.txt": 2,    # Xbox power cable fire fear
    "130.txt": 2,    # Slim PlayStation triples sales
    "170.txt": 2,    # Casual gaming to take off
    "178.txt": 2,    # Slimmer PlayStation triple sales
    "221.txt": 2,    # Halo 2 heralds traffic explosion

    # Digital Technology (0)
    "047.txt": 0,    # Millions buy MP3 players in US
    "061.txt": 0,    # Concern over RFID tags
    "321.txt": 0,    # Consumer concern over RFID tags (dup)

    # Cybersecurity Online (1)
    "284.txt": 1,    # Hacker threat to Apple iTunes

    # Internet Society (3)
    "044.txt": 3,    # Apple attacked over sources row
    "248.txt": 3,    # Apple sues to stop product leaks
    "259.txt": 3,    # Apple sues Tiger file sharers
    "355.txt": 3,    # Apple makes blogs reveal sources

    # Digital Technology (0) — MP3/hardware crossover
    "148.txt": 0,    # iTunes user sues Apple over iPod
}

# ── Build initial DataFrame
final_df_tech = pd.DataFrame({
    'filename':         unique_files,
    'category':         'Tech',
    'sub_category_id':  topics,
    'sub_category':     [tech_labels.get(t, 'Unassigned') for t in topics],
    'confidence_score': [round(float(p), 4) if p > 0 else 0.0 for p in probs],
    'preview':          unique_Tech
})

def tech_remap(row):
    sid = row['sub_category_id']
    if sid == -1:
        sid = tech_noise_remap.get(row['filename'], -1)
    sid = tech_consolidation.get(sid, sid)
    return sid, tech_labels.get(sid, 'Unassigned')

final_df_tech[['sub_category_id', 'sub_category']] = final_df_tech.apply(
    tech_remap, axis=1, result_type='expand'
)

print("\nBBC TECH — SUB-CATEGORY SUMMARY (fully labelled)")
print("=" * 65)

summary = final_df_tech.groupby(['sub_category_id', 'sub_category']).agg(
    article_count=('filename', 'count'),
    avg_confidence=('confidence_score', 'mean')
).reset_index()
summary['avg_confidence'] = summary['avg_confidence'].round(4)
summary = summary.sort_values('sub_category_id')

for _, row in summary.iterrows():
    print(f"\n  [{int(row['sub_category_id'])}] {row['sub_category']}")
    print(f"       Articles: {row['article_count']} | Avg Confidence: {row['avg_confidence']}")

print(f"\n{'=' * 65}")
print(f"Total articles    : {len(final_df_tech)}")
print(f"Sub-categories    : {final_df_tech['sub_category_id'].nunique()}")
print(f"Unassigned (noise): {(final_df_tech['sub_category_id'] == -1).sum()}")

display(final_df_tech[['filename', 'category', 'sub_category', 'confidence_score', 'preview']])